# Sumativa 3 - Punto 2 - Regresión Logística (Versión 6.0)
**Correcciones exactas de Jorge aplicadas:**
- Eliminación de `train_test_split` y manual de atraso (viene del parquet)
- Cálculo exacto de edad: `anio_ing_carr_ori - (fec_nac_alu // 100)`
- Uso de la columna `nomb_inst`
- Bootstrap (seed 42) con **contador de iteraciones fallidas**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Cargar datos de Fernanda (Punto 1 de Jorge)
ruta_train = "../resultados/train_e3.parquet"
ruta_test = "../resultados/test_e3.parquet"

df_train = pd.read_parquet(ruta_train)
df_test = pd.read_parquet(ruta_test)

print(f"Datos cargados: Train {df_train.shape} | Test {df_test.shape}")

In [ ]:
# 2. Edad de ingreso (Punto 2 de Jorge)
# La edad se calcula como: anio_ing_carr_ori - (fec_nac_alu // 100)
if 'anio_ing_carr_ori' in df_train.columns and 'fec_nac_alu' in df_train.columns:
    df_train['edad_ingreso'] = df_train['anio_ing_carr_ori'] - (df_train['fec_nac_alu'] // 100)
    df_test['edad_ingreso'] = df_test['anio_ing_carr_ori'] - (df_test['fec_nac_alu'] // 100)
    print("Edad de ingreso calculada correctamente.")
else:
    print("Faltan las columnas 'anio_ing_carr_ori' o 'fec_nac_alu'.")

In [ ]:
# 3. Filtro de Institución (Punto 3 de Jorge)
if 'nomb_inst' in df_train.columns:
    # Ajusta los nombres válidos según tu pauta, ejemplo genérico:
    instituciones_validas = ['Universidad Andrés Bello', 'Otra Universidad']
    df_train = df_train[df_train['nomb_inst'].isin(instituciones_validas)]
    df_test = df_test[df_test['nomb_inst'].isin(instituciones_validas)]
    print("Filtro aplicado por 'nomb_inst'.")
elif 'cod_inst' in df_train.columns:
    print("Usando 'cod_inst' como alternativa.")
else:
    print("No se encontró 'nomb_inst' ni 'cod_inst'.")

In [ ]:
# Preparación de X e Y (El target es 'atraso_alto' según punto 1 de Jorge)
var_objetivo = 'atraso_alto'
columnas_X = [col for col in df_train.columns if df_train[col].dtype in [np.float64, np.int64] and col != var_objetivo]

X_train = df_train[columnas_X]
y_train = df_train[var_objetivo]

X_test = df_test[columnas_X]
y_test = df_test[var_objetivo]

X_train_procesado = sm.add_constant(X_train)
X_test_procesado = sm.add_constant(X_test)

# PUNTO 6: Modelo 1 (Variables S1/S2 sin edad)
cols_modelo_1 = [col for col in X_train_procesado.columns if 'edad' not in col.lower()]

try:
    modelo_base = sm.Logit(y_train, X_train_procesado[cols_modelo_1]).fit(disp=False)
    print("\n--- Resumen Modelo Base ---")
    print(modelo_base.summary())
except Exception as e:
    print(f"Error en modelo base: {e}")

In [ ]:
# 4. Bootstrap con contador de fallos (Punto 4 de Jorge)
n_iteraciones = 100
coeficientes_boot = []
iteraciones_fallidas = 0

print(f"\nIniciando Bootstrap (n={n_iteraciones}) con semilla 42...")
np.random.seed(42)

for i in range(n_iteraciones):
    X_res, y_res = resample(X_train_procesado[cols_modelo_1], y_train, random_state=42 + i)
    
    try:
        modelo_logit = sm.Logit(y_res, X_res)
        resultado = modelo_logit.fit(disp=0)
        coeficientes_boot.append(resultado.params)
    except (np.linalg.LinAlgError, Exception):
        iteraciones_fallidas += 1
        continue

print(f"\nBootstrap completado.")
print(f"-> Iteraciones exitosas: {len(coeficientes_boot)}")
print(f"-> Iteraciones fallidas: {iteraciones_fallidas}")

if coeficientes_boot:
    df_coeficientes = pd.DataFrame(coeficientes_boot)
    print("\nPromedio de coeficientes:")
    print(df_coeficientes.mean())
